<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook D02: Recurrent Networks</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook D02: Recurrent Networks](../notebooks/D02_Recurrent_networks.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The windows, the model and the training loop from the notebook, with two small generalisations that the
exercises need: `make_windows` takes a `horizon` and optional extra input `channels`, and
`RecurrentForecaster` takes an `input_size`.

`FULL_RUN` works exactly as it does in the notebook. The numbers quoted in the answers come from the
default workshop run, which trains four LSTMs in about four minutes. Setting `FULL_RUN = True` lowers every
error by roughly a fifth without changing any of the conclusions here.

In [ ]:
import sys
import importlib.util
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}, using {torch.get_num_threads()} thread")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK = 168
HORIZON = 24

FULL_RUN = False
TRAIN_STRIDE = 1 if FULL_RUN else 2
EPOCHS = 6

values = load.values.astype(np.float32)
n_observations = len(values)

TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop, stride=1, horizon=HORIZON, offset=0, channels=()):
    """Windows of `LOOKBACK` steps, targeting `horizon` steps from `offset` hours ahead.

    `channels` are extra series aligned with `scaled`, stacked as additional
    input channels alongside the load itself.
    """
    positions = range(start, stop, stride)

    inputs = [np.stack([scaled[t - LOOKBACK:t] for t in positions])[:, :, None]]
    for channel in channels:
        inputs.append(np.stack([channel[t - LOOKBACK:t] for t in positions])[:, :, None])

    targets = np.stack([scaled[t + offset:t + offset + horizon] for t in positions])

    return torch.tensor(np.concatenate(inputs, axis=2)), torch.tensor(targets)


def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


class RecurrentForecaster(nn.Module):
    def __init__(self, kind="LSTM", hidden_size=64, num_layers=1,
                 horizon=HORIZON, input_size=1):
        super().__init__()
        self.recurrent = getattr(nn, kind)(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
        )
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        output, _ = self.recurrent(x)
        return self.head(output[:, -1])


def train_recurrent(data, horizon, kind="LSTM", hidden_size=64, epochs=None,
                    batch_size=256, learning_rate=1e-3, seed=0):
    """Train on (X_train, y_train), keeping the weights that score best on validation."""
    epochs = EPOCHS if epochs is None else epochs
    X_train, y_train, X_validation, y_validation, X_test, y_test = data

    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = RecurrentForecaster(
        kind, hidden_size, horizon=horizon, input_size=X_train.shape[2]
    )
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae, "epoch": epoch,
                    "weights": {k: v.clone() for k, v in model.state_dict().items()}}

    model.load_state_dict(best["weights"])
    model.eval()

    with torch.no_grad():
        predictions = model(X_test).numpy()

    return {"predictions": predictions, "test_mae": score(predictions, y_test.numpy()),
            "validation_mae": best["validation_mae"],
            "parameters": sum(p.numel() for p in model.parameters()),
            "seconds": time.time() - started}


def build_split(horizon=HORIZON, offset=0, channels=()):
    """Train, validation and test windows, with the notebook's boundaries."""
    return (
        *make_windows(scaled, LOOKBACK, train_end - HORIZON, TRAIN_STRIDE,
                      horizon, offset, channels),
        *make_windows(scaled, train_end, train_end + VALIDATION_HOURS - HORIZON,
                      1, horizon, offset, channels),
        *make_windows(scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON,
                      1, horizon, offset, channels),
    )


if TORCH_AVAILABLE:
    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])

    day_ahead = build_split()
    actual_test = to_original_units(day_ahead[5].numpy())
    NAIVE_MAE = mean_absolute_error(actual_test.ravel(), naive_forecast.ravel())

    print(f"{'FULL' if FULL_RUN else 'WORKSHOP'} run, training stride {TRAIN_STRIDE}")
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Retrain the LSTM with `HORIZON = 1` and compare its one-step error against the first column of the table above. Does a model trained only to predict the next hour beat the 24-hour model at that one task, and what would that imply about training separate models per horizon?

In [ ]:
if TORCH_AVAILABLE:
    shared = train_recurrent(day_ahead, HORIZON)
    shared_per_horizon = [
        mean_absolute_error(actual_test[:, h], to_original_units(shared["predictions"])[:, h])
        for h in range(HORIZON)
    ]

    # One model trained only on the next hour
    next_hour = train_recurrent(build_split(horizon=1), 1)

    # And, for contrast, one trained only on the 24th hour ahead
    hour_24 = train_recurrent(build_split(horizon=1, offset=HORIZON - 1), 1)

    single_horizon = pd.DataFrame({
        "24-output model": [shared_per_horizon[0], shared_per_horizon[23]],
        "Dedicated model": [next_hour["test_mae"], hour_24["test_mae"]],
        "Naive": [
            mean_absolute_error(actual_test[:, 0], naive_forecast[:, 0]),
            mean_absolute_error(actual_test[:, 23], naive_forecast[:, 23]),
        ],
    }, index=["1 hour ahead", "24 hours ahead"])

    single_horizon["dedicated wins by"] = (
        1 - single_horizon["Dedicated model"] / single_horizon["24-output model"]
    )

    print("Test MAE in MW\n")
    print(single_horizon.round({"24-output model": 1, "Dedicated model": 1,
                                "Naive": 1, "dedicated wins by": 3}).to_string())

**Yes, and by a wide margin at one hour ahead: 152.9 MW against 267.9, an improvement of 43%.**

A model that does nothing but predict the next hour is dramatically better at predicting the next hour
than a model that also has to predict the following twenty-three. That much the exercise anticipates.

**The second row is the part worth the extra training run.** Train another dedicated model on the 24th
hour ahead alone, and it scores 538.5 against the shared model's 549.9 — an improvement of **2%**.

So the advantage of specialising is not a property of specialisation. It is concentrated almost entirely
at the near horizon, and by 24 hours out it has essentially vanished.

The reason follows from what the shared model is optimising. Its loss is the mean squared error
averaged over all 24 outputs, and those outputs are wildly unequal: the far horizons carry errors around
550 MW, the near ones around 270. **The hard, distant hours dominate the gradient.** The representation
the LSTM settles on is therefore tuned for them, and it is a poor use of a 64-unit hidden state for a task
that mostly needs the last few hours of the trajectory.

Specialising at h = 24 gains nothing for the same reason in reverse: the shared model was already, in
effect, a horizon-24 model. There was nothing left to reclaim.

**What this implies about training separate models per horizon** is less encouraging than the 43% makes it
look:

- **The cost is linear and the benefit is not.** Twenty-four dedicated models cost 24 times the training
  time, 24 sets of weights to store and version, and 24 things to retrain when the data drifts. The
  benefit, measured here, is large at h = 1, negligible at h = 24, and somewhere in between for the hours
  between.
- **The comparison to make first is against the baseline.** At 24 hours ahead every model in that table is
  within 5% of each other and all of them are close to the naive forecast's 563 MW. Arguing about
  architecture at that horizon is arguing about the wrong thing; Exercise 2 shows what actually helps
  there.
- **The middle path usually wins.** Rather than 1 or 24 models, group the horizons: one model for 1–6
  hours, one for 7–24. Each still shares statistical strength across similar tasks, and neither is forced
  to compromise between tasks that want different representations.

There is also a cheaper explanation available for part of the gap, and it should be ruled out before
drawing architectural conclusions: the dedicated model produces one output from the same 64-unit state, so
it has an easier job in a trivial sense as well as a statistical one. The honest version of the finding is
**"multi-horizon output costs the near horizons a lot and the far horizons almost nothing"**, which is a
claim about this data and this horizon range, not a general law.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Add the hour of day and day of week as two extra input channels, so `X` has shape `(batch, 168, 3)` instead of `(batch, 168, 1)`. The network can currently only infer the calendar from the shape of the sequence. How much does telling it directly help?

In [ ]:
hour_of_day = ((load.index.hour.values / 23.0) * 2 - 1).astype(np.float32)
day_of_week = ((load.index.dayofweek.values / 6.0) * 2 - 1).astype(np.float32)

if TORCH_AVAILABLE:
    with_calendar = build_split(channels=(hour_of_day, day_of_week))
    print(f"input shape {tuple(day_ahead[0].shape)} -> {tuple(with_calendar[0].shape)}")

    calendar = train_recurrent(with_calendar, HORIZON)

    calendar_per_horizon = [
        mean_absolute_error(actual_test[:, h], to_original_units(calendar["predictions"])[:, h])
        for h in range(HORIZON)
    ]

    channels_table = pd.DataFrame({
        "Load only": [shared["test_mae"], shared_per_horizon[0], shared_per_horizon[23],
                      shared["parameters"]],
        "+ hour, day of week": [calendar["test_mae"], calendar_per_horizon[0],
                                calendar_per_horizon[23], calendar["parameters"]],
    }, index=["Overall MAE", "MAE at 1 hour", "MAE at 24 hours", "Parameters"])

    print()
    print(channels_table.round(1).to_string())
    print()
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(12, 4.5))

    hours_ahead = np.arange(1, HORIZON + 1)
    ax.plot(hours_ahead, shared_per_horizon, marker="o", markersize=4, linewidth=1.5,
            color="steelblue", label="LSTM, load only")
    ax.plot(hours_ahead, calendar_per_horizon, marker="o", markersize=4, linewidth=1.5,
            color="seagreen", label="LSTM, + calendar channels")
    ax.plot(hours_ahead,
            [mean_absolute_error(actual_test[:, h], naive_forecast[:, h]) for h in range(HORIZON)],
            marker="o", markersize=4, linewidth=1.5, color="crimson", label="Naive")

    ax.fill_between(hours_ahead, calendar_per_horizon, shared_per_horizon,
                    color="seagreen", alpha=0.12)

    ax.set_title("Where the calendar channels help", fontsize=13, fontweight="bold")
    ax.set_xlabel("Hours ahead")
    ax.set_ylabel("MAE (MW)")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**A great deal: 433.5 MW to 305.2, a 30% reduction, for 512 extra parameters.**

That is by far the largest improvement anywhere in this notebook. For comparison, the entire difference
between a plain RNN, a GRU and an LSTM was a fraction of it, and adding a second LSTM layer was worth less
still. **Two columns of arithmetic beat every architectural change in the notebook, combined.**

The shaded region in the plot shows where the gain comes from, and it is the opposite of Exercise 1:

| | load only | + calendar | improvement |
|---|---|---|---|
| **1 hour ahead** | 267.9 | 221.8 | 17% |
| **24 hours ahead** | 549.9 | 358.7 | 35% |

**The calendar helps most exactly where the recent past helps least.** One hour ahead, the last few
observations already tell the network nearly everything — it does not need to be told the time, because
the current load level implies it. Twenty-four hours ahead, the trajectory has stopped being informative,
and the question becomes "what does 7 a.m. on a Tuesday usually look like?" That is a calendar question,
and until now the network had to answer it by counting 168 steps back through a recurrent state.

In [ ]:
# The channels above are linear ramps, so midnight is a seam: hour 23 sits at
# +1.0 and hour 0 at -1.0 despite being adjacent. The cyclical encoding from C01
# has no seam, at the cost of two more channels.
angle_hour = 2 * np.pi * load.index.hour.values / 24
angle_day = 2 * np.pi * load.index.dayofweek.values / 7

cyclical_channels = tuple(
    series.astype(np.float32)
    for series in (np.sin(angle_hour), np.cos(angle_hour),
                   np.sin(angle_day), np.cos(angle_day))
)

if TORCH_AVAILABLE:
    cyclical = train_recurrent(build_split(channels=cyclical_channels), HORIZON)

    print(f"linear ramps   (3 channels)  {calendar['test_mae']:.1f} MW")
    print(f"sine / cosine  (5 channels)  {cyclical['test_mae']:.1f} MW")
    print(f"load only      (1 channel)   {shared['test_mae']:.1f} MW")

It is worth being precise about what the network could and could not do before. The information was
never absent: the input window is 168 consecutive hours, so the *phase* of the daily and weekly cycle is
fully determined by the shape of the sequence. In principle the LSTM can recover it.

In practice it must **learn to count**, maintaining a phase estimate across 168 recurrent steps and
keeping it stable through a hidden state that is simultaneously tracking the load level. That is a hard
thing to ask of 64 units and six epochs, and it is being asked on every single forward pass, for
information that is free.

This is the same lesson as the Fourier terms in Notebook
[C01](../notebooks/C01_Feature_engineering.ipynb), and it generalises past neural networks:

> **If a quantity is known exactly, supply it. Do not make the model infer it.**

Capacity spent rediscovering the calendar is capacity not spent on the part of the problem that is
genuinely hard.

Two footnotes on the implementation, both worth knowing:

- **The encoding barely matters here**, as the cell above measures. The sine/cosine version scores 303.6
  against the linear ramps' 305.2 — indistinguishable, next to the 128 MW that adding the calendar at all
  was worth. The network has 168 steps of context and learns to work around the midnight seam. Use
  cyclical encodings by default, since they cost nothing and cannot hurt, but do not expect this
  particular refinement to be where your accuracy is.
- **These channels are known in advance**, exactly like `open` and `promo` in Notebook
  [D01](./D01_Neural_networks_intro_solutions.ipynb). The hour and weekday of every future timestamp are
  arithmetic. Adding a weather forecast as a fourth channel would be the natural next step and would
  probably help more than anything else in this notebook — but weather is a *forecast*, with its own error,
  and the honest evaluation would have to use the forecast that was available at the time rather than what
  the weather turned out to be. That distinction is the whole subject of Notebook
  [C01](../notebooks/C01_Feature_engineering.ipynb)'s section 3.

---

Back to [Notebook D02](../notebooks/D02_Recurrent_networks.ipynb), or on to
[Notebook D03](../notebooks/D03_Convolutional_networks.ipynb).